# Data Ingestion 



In [1]:
# document data structure
'''
LangChain document structure consist of following components:
- page content
- metadata

document loaders
- pdf loader
- csv loader
- webbase loader
....
''' 

from langchain_core.documents import Document




In [2]:
doc = Document(
    page_content = 'Using this to create RAGs',
    metadata={
        'souirce':'example.txt',
        'pages': 1,
        'author':'zys',
        'date_created':'2026-01-01'
    }
)
doc

Document(metadata={'souirce': 'example.txt', 'pages': 1, 'author': 'zys', 'date_created': '2026-01-01'}, page_content='Using this to create RAGs')

In [3]:
# text file

import os
os.makedirs('../data/text_files', exist_ok=True)

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """
}

for file_path, content in sample_texts.items():
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)

print('sample text generated and saved in data/text_files foleder')

sample text generated and saved in data/text_files foleder


In [5]:
# text loader
# from langchain.document_loaders import TextLoader

from langchain_community.document_loaders.text import TextLoader

loader = TextLoader('../data/text_files/python_intro.txt', encoding='utf-8')
document = loader.load()
print(document)

C:\Users\YashPrasad\AppData\Local\Temp\ipykernel_14356\48700658.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
c:\New_Desktop\COde\GENAI\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [7]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders.pdf import PyPDFLoader, PyMuPDFLoader
## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0', 'creationdate': '2026-03-24T06:32:02+00:00', 'source': '..\\data\\pdf\\C4 Views - Overview.pdf', 'file_path': '..\\data\\pdf\\C4 Views - Overview.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': 'C4 Views - Overview', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-03-24T06:32:02+00:00', 'trapped': '', 'modDate': "D:20260324063202+00'00'", 'creationDate': "D:20260324063202+00'00'", 'page': 0}, page_content='Last updated by | Leonardo Pinto | Nov 24, 2025 at 11:07 PM GMT+5:30\nC4 Views\nOverview\nDo not change any names generated by the transpiler, except for the target schema.\nSetup\n1. \n2. \n3. \n4. \nThe name of the source data mart must be the name of the folder in the transpiled code.\nThe other attributes must be:\ntype: generic\ncompute: serverless\nworkflow trigger: no\n5.

In [8]:
type(pdf_documents[0])

langchain_core.documents.base.Document

# RAG pipeline : Data ingestion to Vector DB

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
def process_pdfs(directory: str):
    all_documents = []
    pdf_dir = Path(directory)

    pdf_files= list(pdf_dir.glob('**/*.pdf'))

    print(f'Found {len(pdf_files)} pdf files in directory')

    for pdf_file in pdf_files:
        print(f'In processing: {pdf_file.name}')
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f'Loaded {len(documents)} pages')
        
        except Exception as e:
            print('Error: ', e)
    
    print(f'\nTotal documents loaded: {len(all_documents)}')
    return all_documents

In [11]:
all_pdf_docs = process_pdfs('../data')

Found 2 pdf files in directory
In processing: C4 Views - Overview.pdf


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 41 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 48 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 63 0 (offset 0)
Ignoring wrong pointing object 75 0 (offset 0)
Ignoring wrong pointing object 95 0 (offset 0)
Ignoring wrong 

Loaded 3 pages
In processing: Yash Prasad 229309105 Major Project Report V2.pdf
Loaded 35 pages

Total documents loaded: 38


In [12]:
all_pdf_docs

[Document(metadata={'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0', 'creationdate': '2026-03-24T06:32:02+00:00', 'title': 'C4 Views - Overview', 'moddate': '2026-03-24T06:32:02+00:00', 'source': '..\\data\\pdf\\C4 Views - Overview.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'C4 Views - Overview.pdf', 'file_type': 'pdf'}, page_content='Last updated by | Leonardo Pinto | Nov 24, 2025 at 11:07 PM GMT+5:30\nC4 Views\nOverview\nDo not change any names generated by the transpiler, except for the target schema.\nSetup\n1. \n2. \n3. \n4. \nThe name of the source data mart must be the name of the folder in the transpiled code.\nThe other attributes must be:\ntype: generic\ncompute: serverless\nworkflow trigger: no\n5. \n6. \n7. \n8. \n9. \nChange the type of the entity to view\xa0 in the .config.yml file.\nThe C4 views are built on the top of the Igni

In [13]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']  
    )

    split_docs = text_splitter.split_documents(documents)
    print(f'split: {len(documents)} documents into {len(split_docs)} chunks')

    if split_docs:
        print('\nExample chunk:')
        print(f'Content: {split_docs[0].page_content[:200]}...')
        print(f'Metadata: {split_docs[0].metadata}')
    
    return split_docs


In [14]:
chunks =split_documents(all_pdf_docs)

split: 38 documents into 128 chunks

Example chunk:
Content: Last updated by | Leonardo Pinto | Nov 24, 2025 at 11:07 PM GMT+5:30
C4 Views
Overview
Do not change any names generated by the transpiler, except for the target schema.
Setup
1. 
2. 
3. 
4. 
The name...
Metadata: {'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0', 'creationdate': '2026-03-24T06:32:02+00:00', 'title': 'C4 Views - Overview', 'moddate': '2026-03-24T06:32:02+00:00', 'source': '..\\data\\pdf\\C4 Views - Overview.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'C4 Views - Overview.pdf', 'file_type': 'pdf'}


In [15]:
chunks

[Document(metadata={'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0', 'creationdate': '2026-03-24T06:32:02+00:00', 'title': 'C4 Views - Overview', 'moddate': '2026-03-24T06:32:02+00:00', 'source': '..\\data\\pdf\\C4 Views - Overview.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'C4 Views - Overview.pdf', 'file_type': 'pdf'}, page_content="Last updated by | Leonardo Pinto | Nov 24, 2025 at 11:07 PM GMT+5:30\nC4 Views\nOverview\nDo not change any names generated by the transpiler, except for the target schema.\nSetup\n1. \n2. \n3. \n4. \nThe name of the source data mart must be the name of the folder in the transpiled code.\nThe other attributes must be:\ntype: generic\ncompute: serverless\nworkflow trigger: no\n5. \n6. \n7. \n8. \n9. \nChange the type of the entity to view\xa0 in the .config.yml file.\nThe C4 views are built on the top of the Igni

# Embeddings and VectorStoreDB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class EmbeddingManager:
    '''Handles document embedding generation using SentenceTransformer'''
    
    def __init__(self, model_name: str='all-MiniLM-L6-v2'):
        '''
        Initialize embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        '''
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        '''Load the SentenceTransformer model'''
        try:
            print(f'Loading embedding model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f'Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}')
        
        except Exception as e:
            print(f'Error loading model {self.model_name}: {e}')
            raise 
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        '''
        Generate embeddings for a list of texts

        Arg:
        texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        '''
        print(f'Generatng embeddings for {len(texts)} texts....')
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f'Generated embeddings with shape: {embeddings.shape}')

        return embeddings

    def get_embedding_dimension(self) -> int:
        '''Get the embedding dimension of the model'''
        if not self.model:
            raise ValueError('Model not loaded')
        return self.model.get_embedding_dimension()
    

# initialize the embedding manager
embeddings_manager = EmbeddingManager()
embeddings_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9366.14it/s]


Model loaded successfully. Embedding dimension: 384


# Vector Store

In [25]:
class VectorStore:
    '''Manages document embedding in a ChromaDB Vector store'''

    def __init__(self, collection_name: str='pdf_documents', persist_directory: str='../data/vector_store'):
        '''
        Initialise the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        '''

        self.collection_name = collection_name
        self.persist_directory=persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

        
    def _initialize_store(self):
        '''Initialise chromaDB client and collections'''
        try:
            # persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # get or create collections
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'description': 'PDF document embeddings for RAG'}                
            )

            print(f'Vectore store initialised. Collection: {self.collection_name}')
            print(f'Existing documents in collection: {self.collection.count()}')
        
        except Exception as e:
            print('Error initializing vector store: {e}')
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        '''
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        '''
        if len(documents) != len(embeddings):
            raise ValueError('Number of documents must match number of embeddings')
        print(f'Adding {len(documents)} documents of vector store...')

        # data preparation for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # generating unique id
            doc_id = f'doc_{uuid.uuid4().hex[:8]}_{i}'
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # embeddings
            embeddings_list.append(embedding.tolist())
        
        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f'Successfully added {len(documents)} documents to vector store')
            print(f'Total documents in collection: {self.collection.count()}')
        
        except Exception as e:
            print(f'Error adding documents to vector store: {e}')
            raise

vectorstore = VectorStore()
vectorstore



Vectore store initialised. Collection: pdf_documents
Existing documents in collection: 0


In [26]:
chunks

[Document(metadata={'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0', 'creationdate': '2026-03-24T06:32:02+00:00', 'title': 'C4 Views - Overview', 'moddate': '2026-03-24T06:32:02+00:00', 'source': '..\\data\\pdf\\C4 Views - Overview.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'C4 Views - Overview.pdf', 'file_type': 'pdf'}, page_content="Last updated by | Leonardo Pinto | Nov 24, 2025 at 11:07 PM GMT+5:30\nC4 Views\nOverview\nDo not change any names generated by the transpiler, except for the target schema.\nSetup\n1. \n2. \n3. \n4. \nThe name of the source data mart must be the name of the folder in the transpiled code.\nThe other attributes must be:\ntype: generic\ncompute: serverless\nworkflow trigger: no\n5. \n6. \n7. \n8. \n9. \nChange the type of the entity to view\xa0 in the .config.yml file.\nThe C4 views are built on the top of the Igni

In [28]:
# convert text to embeddings
texts = [doc.page_content for doc in chunks]

# generate the embeddings
embeddings = embeddings_manager.generate_embeddings(texts)

# store in vetor database 
vectorstore.add_documents(chunks, embeddings)

Generatng embeddings for 128 texts....


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]


Generated embeddings with shape: (128, 384)
Adding 128 documents of vector store...
Successfully added 128 documents to vector store
Total documents in collection: 128


# Retriever Pipeline from Vector Store

In [ ]:
class RAGRetriever:
    '''Handles query-based retrieval from the vector store'''

    def __init__(self, vector_store: VectorStore, embeddings_manager: EmbeddingManager):
        '''
        Initialise the retriever

        Args:
            vector_store: Vector stroe containing document embeddings
            embedding_manager: Manager for generating query embeddings
        '''

        self.vector_store=vector_store
        self.embedding_manager = embeddings_manager
    
    def retrieve(self, query: str, top_k: int=5, score_threshold: float=0.0) -> List[Dict[str, Any]]:
        '''
        Retrieve relevant documents for a query

        Args:
            query: The search query
            tok_k: Number of top results to return 
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of dictionaries containing retrieved documents and metadata
        '''
        print()